In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
from tqdm import tqdm
import re
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, precision_score, recall_score 
import lightgbm as lgb
import pickle
from sklearn.preprocessing import LabelEncoder
import datetime

In [3]:
CSV_PATH = "../csv/"
K_COLS = ['着', 'RaceID', '選手登番', '展示', 'スタートタイミング']
DATE = "221002"

In [6]:
k_files = glob.glob("../csv/K" + DATE + ".csv")
b_files = glob.glob("../csv/B" + DATE + ".csv")

In [7]:
def concat_files(files):
    tmp = pd.DataFrame()
    for file in tqdm(files):
        df = pd.read_csv(file, index_col=0)
        tmp = pd.concat([tmp,df])
    return tmp

In [8]:
kdf = concat_files(k_files)
bdf = concat_files(b_files)

100%|██████████| 1/1 [00:00<00:00, 51.88it/s]


In [9]:
df = pd.merge(bdf,kdf.loc[:,K_COLS],on = ['RaceID','選手登番'])

In [10]:
def LabelEncoding(df,col):
    #インスタンス
    LE = LabelEncoder()

    #Label エンコーディング
    LE.fit_transform(df[col].values)

    pickle.dump(LE, open('../model/' + col + '_LE.pickle', 'wb'))

    #データ変換
    le = LE.fit_transform(df[col].values)
    return le

def zero_padding(txt):
    l = re.findall(r"\d+", txt)
    l = [int(s) for s in l]
    date = datetime.date(*l[:3])
    raceid = str(date) + '-' + str(l[3]).zfill(2) + '-' + str(l[4]).zfill(2)
    return raceid

In [14]:
df_encoded = df
df_encoded['支部'] = LabelEncoding(df,'支部')
df_encoded['級別'] = LabelEncoding(df, '級別')
df_encoded['R'] = df['R'].replace('R', '', regex=True).astype('int')
df_encoded['RaceID'] = df_encoded['RaceID'].replace('_','/',regex=True).replace('R','',regex=True)

In [15]:
zero = []
for n,i in enumerate(tqdm(df_encoded['RaceID'])):
    zero.append(zero_padding(i))
df_encoded['RaceID'] = zero

100%|██████████| 990/990 [00:00<00:00, 104995.47it/s]


In [21]:
df_encoded['RaceID']

0      2022-10-02-23-01
1      2022-10-02-23-01
2      2022-10-02-23-01
3      2022-10-02-23-01
4      2022-10-02-23-01
             ...       
985    2022-10-02-01-12
986    2022-10-02-01-12
987    2022-10-02-01-12
988    2022-10-02-01-12
989    2022-10-02-01-12
Name: RaceID, Length: 990, dtype: object

In [20]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 990 entries, 0 to 989
Data columns (total 19 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   艇番         990 non-null    int64  
 1   選手登番       990 non-null    int64  
 2   選手名        990 non-null    object 
 3   年齢         990 non-null    int64  
 4   支部         990 non-null    int64  
 5   体重         990 non-null    int64  
 6   級別         990 non-null    int64  
 7   全国勝率       990 non-null    float64
 8   全国2連率      990 non-null    float64
 9   当地勝率       990 non-null    float64
 10  当地2連率      990 non-null    float64
 11  モーター2連率    990 non-null    float64
 12  ボート2連率     990 non-null    float64
 13  RaceID     990 non-null    object 
 14  場所         990 non-null    int64  
 15  R          990 non-null    int64  
 16  着          990 non-null    int64  
 17  展示         990 non-null    float64
 18  スタートタイミング  990 non-null    float64
dtypes: float64(8), int64(9), object(2)
memory usage: 1

In [22]:
df_test = df_encoded.drop(['選手名','着','RaceID'],axis=1)

In [23]:
lgb_clf = pickle
with open('../model/lgb_clf.pickle', 'rb') as f:
    lgb_clf = pickle.load(f)

In [25]:
df_pred = lgb_clf.predict(df_test ,num_iteration=lgb_clf.best_iteration)

In [31]:
df_encoded['着'][:12]

0     1
1     2
2     5
3     6
4     4
5     3
6     4
7     1
8     3
9     6
10    5
11    2
Name: 着, dtype: int64

In [30]:
df_pred

array([-1.22319862e+00, -3.46896964e-01,  2.40842018e+00,  6.77114440e-01,
        1.00715025e+00,  1.26276806e+00, -1.91543287e+00, -1.86799524e+00,
        8.77558875e-01,  1.79164652e+00,  1.41442150e+00,  9.06552065e-01,
       -2.37824215e+00,  2.12486071e-01, -7.37158516e-01, -1.14134747e+00,
        3.17861198e-01,  1.15021860e+00, -1.38955262e-02, -1.72068062e+00,
        3.36982679e-01, -3.08286742e-01, -4.55852036e-01,  1.58957978e+00,
        4.77100414e-01,  1.06392349e+00,  1.12155565e+00, -1.71475960e+00,
       -9.35468668e-01,  1.74049699e+00,  1.15108422e+00,  1.45583606e-01,
        8.28323487e-01, -2.26420740e+00, -1.14441682e+00,  1.77255253e-01,
        1.24730904e+00,  1.69217100e+00,  1.89818946e+00, -1.48338397e+00,
       -2.69871917e-01,  2.01594861e+00,  1.41732997e+00,  3.48577572e-01,
        5.75344126e-01,  1.10262569e+00,  2.87558495e-01, -2.03278574e-01,
       -1.60465329e+00,  1.61548917e-01, -1.31870903e+00, -2.55482790e-01,
       -6.90525570e-01, -